In [2]:
import pandas as pd
df_tickers = pd.read_csv('../strategy/multi_dim_stock_list.csv')
tickers = list(df_tickers.symbol)
long_tickers = list(df_tickers.loc[df_tickers.direction == 'long', 'symbol'])
short_tickers = list(df_tickers.loc[df_tickers.direction == 'short/trend', 'symbol'])
df = pd.read_csv('../analysis/star_1_17_year_holdings.csv')
df.loc[:, 'date1'] =  pd.to_datetime(df['date']).values

In [28]:
import pandas as pd
from eodhd import APIClient
from datetime import datetime, timedelta

# --- Configuration ---
API_KEY = '693327461e9541.04731237' 
START_DATE = "2008-01-21"  # Set this to whenever you actually want to start
SYMBOL = "SP500TR.INDX"       # S&P 500 Index

def get_full_snp_series(api_key, start_date):
    client = APIClient(api_key)
    today_str = datetime.now().strftime('%Y-%m-%d')
    
    # Explicitly define end_date to override any default '1-year' behavior
    df = client.get_historical_data(
        symbol=SYMBOL, 
        interval="d", 
        iso8601_start=start_date, 
        iso8601_end=today_str
    )
    
    if df.empty:
        return pd.Series(name="No Data", dtype='float64')

    df.index = pd.to_datetime(df.index)
    
    # Logic to grab every 4 weeks
    results = []
    current_target = df.index[0]
    last_date = df.index[-1]
    
    while current_target <= last_date:
        # Get the first available trading day on or after the target
        available_dates = df.index[df.index >= current_target]
        
        if available_dates.empty:
            break
            
        actual_date = available_dates[0]
        results.append(actual_date)
        
        # Increment by exactly 28 days
        current_target = actual_date + timedelta(weeks=4)

    # Clean duplicates and extract the 'close' column as a Series
    final_dates = list(dict.fromkeys(results))
    series = df.loc[final_dates, 'close'].squeeze()
    series.index.name = 'Date'
    
    return series

# --- Execute ---
snp_series = get_full_snp_series(API_KEY, START_DATE)
print(f"Retrieved {len(snp_series)} data points.")
snp_series

Retrieved 239 data points.


Date
2008-01-22    2060.14990000
2008-02-19    2124.12010000
2008-03-18    2100.37990000
2008-04-15    2108.59010000
2008-05-13    2220.46000000
                  ...      
2025-12-30   15332.07030000
2026-01-27   15527.82030000
2026-02-24   15346.74020000
2026-03-24   14620.37990000
2026-04-21   15763.01950000
Name: close, Length: 239, dtype: object

In [29]:
snp_index = snp_series/snp_series.iloc[0]
snp_index

Date
2008-01-22   1.00000000
2008-02-19   1.03105124
2008-03-18   1.01952771
2008-04-15   1.02351295
2008-05-13   1.07781477
                ...    
2025-12-30   7.44221103
2026-01-27   7.53722838
2026-02-24   7.44933182
2026-03-24   7.09675539
2026-04-21   7.65139444
Name: close, Length: 239, dtype: object

In [30]:
import plotly.graph_objects as go 

s_value = pd.Series(dict(zip(df.date1, df.iloc[:, 1:-2].sum(axis=1))))
s_index = s_value/s_value.values[0]

fig = go.Figure()
fig.add_trace(go.Scatter(x = s_index.index, y = s_index.values, name = 'investment index'))
fig.add_trace(go.Scatter(x = snp_index.index, y = snp_index.values, name = 's&p 500 index'))

fig.show()




In [10]:
s_value.notnull().sum()


np.int64(0)

In [57]:
df1 = pd.DataFrame({
        'year': df.date1.dt.year,
        'value': df.loc[:, tickers + ['GIC']].sum(axis = 1)
    })
df2 = df1.groupby('year').agg(lambda x: (x.values[-1]/x.values[0])**(len(x)/13)) - 1
df2

,value
year,
2008,-0.115343
2009,0.174249
2010,0.198904
2011,-0.044504
2012,0.143946
2013,0.426331
2014,0.072613
2015,0.069615
2016,0.314558


In [52]:
df1.loc[df1.year == 2008]

,year,value
0,2008,132388.5782
1,2008,133726.0525
2,2008,124601.4533
3,2008,150215.3729
4,2008,277685.9414
5,2008,124095.5349
6,2008,116838.7188
7,2008,284661.6984
8,2008,170869.0807
9,2008,130890.0079


In [26]:
import plotly.graph_objects as go 

df.loc[:, 'long'] = df[long_tickers].sum(axis = 1)/df[tickers + ['GIC']].sum(axis = 1)
df.loc[:, 'short'] = df[short_tickers].sum(axis = 1)/df[tickers + ['GIC']].sum(axis = 1)
df.loc[:, 'gic'] = df['GIC']/df[tickers + ['GIC']].sum(axis = 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x = df.date, y = df.long, name = 'long'))
fig.add_trace(go.Scatter(x = df.date, y = df.short, name = 'short'))
fig.add_trace(go.Scatter(x = df.date, y = df.gic, name = 'gic'))




In [27]:
import xarray as xr
da_mom = xr.open_dataarray('../simulation_data/momentum.nc')

band = 'dollar_ret_1p'
s_long = da_mom.sel(symbol = long_tickers, band = band).to_pandas().mean(axis = 0)
s_short = da_mom.sel(symbol = short_tickers, band = band).to_pandas().mean(axis = 0)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x = s_long.index, y = s_long.values, name = 'long'))
fig1.add_trace(go.Scatter(x = s_short.index, y = s_short.values, name = 'short'))



fig1.show()

Index(['date', 'VIX_RATIO_SMOOTH', 'YIELD_SPREAD_SMOOTH', 'HY_SPREAD_SMOOTH',
       'FED_RATE_SMOOTH'],
      dtype='str')

In [69]:
df_cons = pd.read_csv('../analysis/conservative_star_17_year_holdings.csv')
df_cons.loc[:, 'date1'] = pd.to_datetime(df_cons['date'])
df1 = pd.DataFrame({
        'year': df_cons.date1.dt.year,
        'value': df_cons.loc[:, tickers + ['GIC']].sum(axis = 1)
    })
df3 = df1.groupby('year').agg(lambda x: (x.values[-1]/x.values[0])**(len(x)/13)) - 1
df3 = df3.rename(columns = {'value': 'conservative'})
df4 = df2.join(df3)
df4.loc[:, 'max'] = df4.max(axis = 1)
df4

,value,conservative,max
year,,,
2008,-0.115343,-0.083662,-0.083662
2009,0.174249,0.214837,0.214837
2010,0.198904,0.217507,0.217507
2011,-0.044504,-0.058703,-0.044504
2012,0.143946,0.124062,0.143946
2013,0.426331,0.305478,0.426331
2014,0.072613,0.057579,0.072613
2015,0.069615,0.005449,0.069615
2016,0.314558,0.211308,0.314558


In [67]:
df4.value < df4.conservative

year
2008     True
2009     True
2010     True
2011    False
2012    False
2013    False
2014    False
2015    False
2016    False
2017    False
2018    False
2019    False
2020    False
2021     True
2022    False
2023    False
2024    False
dtype: bool